In [2]:
import pandas as pd
from IPython.display import display


# Load the dataset
df = pd.read_csv('csv/raw_death_full.csv')


In [3]:

# Take a quick peek at the first 5 rows
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2827 entries, 0 to 2826
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rSerial            2827 non-null   int64  
 1   rID                2827 non-null   object 
 2   rSex               2826 non-null   object 
 3   rDead              2827 non-null   object 
 4   rSage              2494 non-null   float64
 5   rEage              2494 non-null   float64
 6   rAgeRange          2827 non-null   object 
 7   rDeathCity         583 non-null    object 
 8   rDeathAddr         2824 non-null   object 
 9   rPlaceOfDiscovery  2824 non-null   object 
 10  rDteDeathYYY       2749 non-null   float64
 11  rDteDeathMM        2749 non-null   float64
 12  rDteDeathDD        2749 non-null   float64
 13  rTimeOfDiscovery   2749 non-null   object 
 14  rSize              1 non-null      object 
 15  rSHeight           0 non-null      float64
 16  rEHeight           0 non

In [4]:
df.describe()

# DeathYYY has min 11, doesn't make sense

,rSerial,rSage,rEage,rDteDeathYYY,rDteDeathMM,rDteDeathDD,rSHeight,rEHeight,rThings,rDetailsViewed
count,2827.000000,2494.000000,2494.000000,2749.000000,2749.000000,2749.000000,0.0,0.0,0.0,2827.0
mean,240266.579059,31.181636,34.447073,93.538378,6.602765,15.665697,NaN,NaN,NaN,0.0
std,107427.692408,25.877854,27.556488,12.649502,3.255620,8.847564,NaN,NaN,NaN,0.0
min,63.000000,-50.000000,0.000000,11.000000,1.000000,1.000000,NaN,NaN,NaN,0.0
25%,197517.000000,0.000000,0.000000,83.000000,4.000000,8.000000,NaN,NaN,NaN,0.0
50%,256843.000000,32.500000,40.000000,92.000000,7.000000,15.000000,NaN,NaN,NaN,0.0
75%,261009.500000,50.000000,60.000000,104.000000,9.000000,23.000000,NaN,NaN,NaN,0.0
max,451652.000000,662.000000,95.000000,115.000000,12.000000,31.000000,NaN,NaN,NaN,0.0


In [5]:
df.shape

(2827, 33)

In [6]:
df[df['rDteDeathYYY'] <= 20] 

,rSerial,rID,rSex,rDead,rSage,rEage,rAgeRange,rDeathCity,rDeathAddr,rPlaceOfDiscovery,...,rReasonB,rReasonC,rReasonD,rPlace,rUnitID,rUnitName,rNoteUnit,rECaseNo,rDetailsViewed,rHandlingUnit
2748,380283,20240322102253615,男,不詳,NaN,NaN,不詳,屏東縣,屏東縣車城鄉後灣路246號,屏東縣屏東縣車城鄉後灣路246號,...,可能生前落水。,NaN,NaN,屏東市殯,PTN,臺灣屏東地方檢察署,NaN,Z111109BQ241PP7,0,NaN


In [7]:
df.loc[df['rID'] == "20240322102253615", 'rDteDeathYYY'] = 111
df.loc[df['rID'] == "20240322102253615", 'rTimeOfDiscovery'] = '民國111年09月11日'

In [8]:
df['Reason'] = df['rReasonA'].str.cat([df['rReasonB'], df['rReasonC'], df['rReasonD']], sep=' ', na_rep='')

In [9]:
# Apply the +1911 Offset to the Year
# Note: We only add 1911 if the year is already present (not NaN)
df['year_corr'] = df['rDteDeathYYY'] + 1911

# Create the DeathDate column
# pd.to_datetime will return NaT if Year, Month, or Day is missing
df['DeathDate'] = pd.to_datetime(pd.DataFrame({
    'year': df['year_corr'],
    'month': df['rDteDeathMM'],
    'day': df['rDteDeathDD']
}), errors='coerce')

# 4. Generate the Weekday Number (0=Mon, 6=Sun)
df['WeekdayNum'] = df['DeathDate'].dt.weekday

# Strip the time and keep only the YYYY-MM-DD
df['DeathDate'] = df['DeathDate'].dt.date

# 5. Map to Traditional Chinese Weekdays
chinese_weekdays = {
    0: "星期一",
    1: "星期二",
    2: "星期三",
    3: "星期四",
    4: "星期五",
    5: "星期六",
    6: "星期日"
}
df['Weekday'] = df['WeekdayNum'].map(chinese_weekdays)

In [10]:
# Drop the old columns
df.drop(columns=['rDteDeathYYY', 'rDteDeathMM', 'rDteDeathDD', 'rReasonA', 'rReasonB', 'rReasonC', 'rReasonD', 'rDetailsViewed', 'year_corr'], inplace=True)

In [11]:
with pd.option_context('display.max_columns', None):
    display(df)

,rSerial,rID,rSex,rDead,rSage,rEage,rAgeRange,rDeathCity,rDeathAddr,rPlaceOfDiscovery,rTimeOfDiscovery,rSize,rSHeight,rEHeight,rHeightRange,rBody,rDress,rThings,rDeathType,rPlace,rUnitID,rUnitName,rNoteUnit,rECaseNo,rHandlingUnit,Reason,DeathDate,WeekdayNum,Weekday
0,451652,20260331010046691,男,不詳(疑吳文泰),50.0,50.0,約 50 ～ 50 歲,高雄市,高雄市仁武區安樂三街76巷2號2丙房,高雄市高雄市仁武區安樂三街76巷2號2丙房,民國115年03月29日,NaN,NaN,NaN,不詳,NaN,NaN,NaN,未確認,高雄市立殯儀館冰櫃72,QTN,臺灣橋頭地方檢察署,仁武分局,NaN,仁武分局,解剖鑑定。,2026-03-29,6.0,星期日
1,451338,20260325010044184,不詳,不詳,0.0,0.0,不詳,桃園市,桃園市蘆竹區台61南下地磅站1.2號風力發電機組後方海灘,桃園市桃園市蘆竹區台61南下地磅站1.2號風力發電機組後方海灘,民國115年03月23日,NaN,NaN,NaN,不詳,NaN,NaN,NaN,未確認,桃園殯儀館,TYN,臺灣桃園地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2026-03-23,0.0,星期一
2,450940,20260318010045115,男,不詳,63.0,64.0,約 63 ～ 64 歲,宜蘭縣,宜蘭縣五結鄉冬山河親水公園青龍岸,宜蘭縣宜蘭縣五結鄉冬山河親水公園青龍岸,民國115年03月16日,NaN,NaN,NaN,不詳,NaN,NaN,NaN,未確認,員山福園,ILN,臺灣宜蘭地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2026-03-16,0.0,星期一
3,450877,20260317010044301,不詳,不詳,NaN,NaN,不詳,新北市,新北市八里區挖子尾街一號(八里安檢所前海灘上),新北市新北市八里區挖子尾街一號(八里安檢所前海灘上),民國115年03月15日,NaN,NaN,NaN,不詳,NaN,NaN,NaN,未確認,板殯183號,SLN,臺灣士林地方檢察署,NaN,NaN,NaN,暫冰存。,2026-03-15,6.0,星期日
4,450798,20260316010044440,不詳,不詳,50.0,60.0,約 50 ～ 60 歲,基隆市,基隆市中正區平一路148之1號,基隆市基隆市中正區平一路148之1號,民國115年03月14日,NaN,NaN,NaN,不詳,NaN,NaN,NaN,未確認,基隆市立殯儀館,KLN,臺灣基隆地方檢察署,NaN,NaN,NaN,暫冰存。 不得火葬。,2026-03-14,5.0,星期六
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2822,261335,20160812143123784,男,不詳,0.0,0.0,不詳,NaN,真好十二號漁船於東經124度4分北緯28度25分海域撈獲,真好十二號漁船於東經124度4分北緯28度25分海域撈獲,NaN,NaN,NaN,NaN,不詳,NaN,NaN,NaN,NaN,NaN,KLN,臺灣基隆地方檢察署,基隆港務警察總隊,NaN,基隆港務警察總隊,疑溺死。,NaN,NaN,NaN
2823,261336,20160812143851199,男,不詳,0.0,0.0,不詳,NaN,真好十二號漁船於東經124度4分北緯28度25分海域撈獲,真好十二號漁船於東經124度4分北緯28度25分海域撈獲,NaN,NaN,NaN,NaN,不詳,NaN,NaN,NaN,NaN,NaN,KLN,臺灣基隆地方檢察署,基隆港務警察總隊,NaN,基隆港務警察總隊,疑溺死。,NaN,NaN,NaN
2824,261338,20160812144931094,不詳,不詳,0.0,0.0,不詳,NaN,滿福引號漁船於東經125度5分北緯30度30分海域撈獲,滿福引號漁船於東經125度5分北緯30度30分海域撈獲,NaN,NaN,NaN,NaN,不詳,NaN,NaN,NaN,NaN,NaN,KLN,臺灣基隆地方檢察署,基隆港務警察總隊,NaN,基隆港務警察總隊,溺死。,NaN,NaN,NaN
2825,272508,20170313115941100,不詳,不詳,0.0,0.0,不詳,NaN,金門縣烈嶼鄉東崗岸際,金門縣烈嶼鄉東崗岸際,NaN,NaN,NaN,NaN,不詳,NaN,NaN,NaN,未確認,NaN,KMN,福建金門地方檢察署,金門岸巡總隊,NaN,金門岸巡總隊,窒息。 呼吸道溺沒液阻塞。 生前落水。,NaN,NaN,NaN


In [12]:
unique_units = df['rUnitName'].unique()
print(unique_units)

['臺灣橋頭地方檢察署' '臺灣桃園地方檢察署' '臺灣宜蘭地方檢察署' '臺灣士林地方檢察署' '臺灣基隆地方檢察署' '臺灣屏東地方檢察署'
 '臺灣臺北地方檢察署' '臺灣臺南地方檢察署' '臺灣臺東地方檢察署' '臺灣澎湖地方檢察署' '臺灣雲林地方檢察署' '臺灣花蓮地方檢察署'
 '臺灣新北地方檢察署' '臺灣高雄地方檢察署' '臺灣南投地方檢察署' '臺灣新竹地方檢察署' '臺灣臺中地方檢察署' '臺灣彰化地方檢察署'
 '臺灣苗栗地方檢察署' '臺灣嘉義地方檢察署' '福建連江地方檢察署' '福建金門地方檢察署' '法務部法醫研究所']


In [13]:
# Create the 'Region' column

region_map = {
    # 北部 (North)
    '臺灣臺北地方檢察署': '北部', '臺灣新北地方檢察署': '北部', '臺灣士林地方檢察署': '北部',
    '臺灣桃園地方檢察署': '北部', '臺灣新竹地方檢察署': '北部', '臺灣基隆地方檢察署': '北部',
    '臺灣宜蘭地方檢察署': '北部', '法務部法醫研究所': '北部',
    
    # 中部 (Central)
    '臺灣臺中地方檢察署': '中部', '臺灣彰化地方檢察署': '中部', '臺灣南投地方檢察署': '中部',
    '臺灣苗栗地方檢察署': '中部', '臺灣雲林地方檢察署': '中部',
    
    # 南部 (South)
    '臺灣高雄地方檢察署': '南部', '臺灣橋頭地方檢察署': '南部', '臺灣臺南地方檢察署': '南部',
    '臺灣屏東地方檢察署': '南部', '臺灣嘉義地方檢察署': '南部',
    
    # 東部 (East)
    '臺灣花蓮地方檢察署': '東部', '臺灣臺東地方檢察署': '東部',
    
    # 外島 (Outlying Islands)
    '臺灣澎湖地方檢察署': '外島', '福建金門地方檢察署': '外島', '福建連江地方檢察署': '外島',
}

df['Region'] = df['rUnitName'].map(region_map)

# Check if any units were missed (will show up as NaN)
print(df[df['Region'].isna()]['rUnitName'].unique())

[]


In [14]:
import re

#######     Clean up city names      #######

# 1. Modernize the cities in the address and standardize 台 to 臺

cols_to_fix = ['rDeathCity', 'rDeathAddr', 'rPlaceOfDiscovery']

# Apply it to the whole column
df[cols_to_fix] = df[cols_to_fix].replace('台', '臺', regex=True)

# Mapping of old names to new standards
modern_map = {
    '臺中縣': '台中市',
    '臺南縣': '台南市',
    '高雄縣': '高雄市',
    '臺北縣': '新北市',
    '桃園縣': '桃園市',
    '新莊市': '新北市'
}

# Apply the replacement to the address column
df[cols_to_fix] = df[cols_to_fix].replace(modern_map, regex=True)

# 2. Extract city from address

# We define a small helper to handle the logic
def find_admin_unit(text: str):
    if pd.isna(text): 
        return None
    # We look for 2 characters + the marker
    match = re.search(r'(.{2}[市縣區鄉鎮])', str(text))
    return match.group(1) if match else None


df['ExtractedCity'] = df['rDeathAddr'].apply(find_admin_unit)

# Clean up any weird leading/trailing whitespace
df['ExtractedCity'] = df['ExtractedCity'].str.strip()

extracted_compare = df[['rTimeOfDiscovery', 'rDeathCity', 'rDeathAddr', 'ExtractedCity', 'rUnitName']].copy()

df.drop(columns=['ExtractedCity'], inplace=True)

# 3. Fill nan city with extracted cities

# Define the filter (the rows where rDeathCity is null)
mask = (
    extracted_compare['rDeathCity'].isna() |
    (extracted_compare['rDeathAddr'].str.contains('其他', na=False) & 
     extracted_compare['ExtractedCity'].notna())
)

# Use .loc[rows, column] to fill the values directly in the original df
extracted_compare.loc[mask, 'rDeathCity'] = extracted_compare.loc[mask, 'ExtractedCity']

# 4. Fill the rest with the city the unit is in
# Create a mapping that forces the "City/County" suffix
def unit_to_full_city(unit: str):
    if '橋頭' in unit: return '高雄市'
    if '士林' in unit: return '臺北市'
    if '法醫' in unit: return '新北市'
    
    # Most Prosecutors offices are "City" level, but some are "County"
    name = unit[2:4].replace('台', '臺')
    
    # List of known Counties (remaining are usually Cities)
    counties = ['屏東', '宜蘭', '花蓮', '臺東', '澎湖', '彰化', '南投', '苗栗', '雲林', '嘉義', '連江', '金門']
    
    if name in counties:
        return f"{name}縣"
    else:
        return f"{name}市"

extracted_compare.loc[:, 'UnitFullCity'] = extracted_compare['rUnitName'].apply(unit_to_full_city)

extracted_compare.loc[extracted_compare['rDeathCity'].isna(), 'rDeathCity'] = extracted_compare.loc[mask, 'UnitFullCity']

# 5. Finalize cities names with unit cities

# The 22 official top-level administrative divisions
STANDARD_CITY = [
    '臺北市', '新北市', '桃園市', '臺中市', '臺南市', '高雄市',
    '基隆市', '新竹市', '嘉義市', '新竹縣', '苗栗縣', '彰化縣', 
    '南投縣', '雲林縣', '嘉義縣', '屏東縣', '宜蘭縣', '花蓮縣', 
    '臺東縣', '澎湖縣', '金門縣', '連江縣'
]

def finalize_city_name(city_name, unit_name, address, unit_city_name, city_name_set):
  
    # Check if the value is missing, "不詳", "其他", or lacks "市/縣"   
    needs_replace = (
        pd.isna(city_name) or 
        city_name == 'nan' or 
        '不詳' in city_name or 
        '其他' in city_name or 
        (not ('市' in city_name or '縣' in city_name))
    )

    # List of Shilin districts that belong to New Taipei City
    shilin_new_taipei = ['淡水', '八里', '三芝', '石門']

    if needs_replace:
        # --- Special Case: Shilin ---
        if '士林' in unit_name:
            # Look at the address to decide between Taipei or New Taipei
            if any(dist in address for dist in shilin_new_taipei):
                return '新北市'
            else:
                return '臺北市'
        
        # --- General Case ---
        return unit_city_name
    
    # Otherwise (for 鳳山市, 永和市, 剛雄縣, etc.), use the Unit's City
    if city_name not in city_name_set:
        return unit_city_name
    
    # If it already has "市" or "縣" and isn't "不詳/其他", keep it
    return city_name

# Apply to the DataFrame using .loc to avoid the SettingWithCopyWarning
extracted_compare['rDeathCity'] = extracted_compare.apply(
    lambda row: finalize_city_name(
        row['rDeathCity'], 
        row['rUnitName'], 
        row['rDeathAddr'], 
        row['UnitFullCity'],
        STANDARD_CITY
    ), 
    axis=1
)
df.loc[:, 'rDeathCity'] = extracted_compare['rDeathCity']

# Convert both to sets for easy comparison
generated_set = set(df['rDeathCity'].unique())
standard_set = set(STANDARD_CITY)

# 1. Non-standard values currently in your data (The "Clean-up" List)
extra_in_data = generated_set - standard_set

# 2. Standard cities that are completely missing from your data
missing_from_data = standard_set - generated_set

print(f"--- Comparison Results ---")
print(f"Non-standard values found in Data: {sorted(list(extra_in_data))}")
print(f"Standard cities NOT present in Data: {sorted(list(missing_from_data))}")
print(df[df['rDeathCity'].isna()][['rUnitName', 'rDeathAddr']])

--- Comparison Results ---
Non-standard values found in Data: []
Standard cities NOT present in Data: []
Empty DataFrame
Columns: [rUnitName, rDeathAddr]
Index: []


In [15]:
######      Creating the Unit-to-Region Mapping DataFrame        #######

# 1. Select the columns and drop duplicates to get unique pairs
unit_region_df = df[['rUnitName', 'Region']].drop_duplicates().reset_index(drop=True)

# 2. Define the Sort Order mapping
order_map = {
    '北部': 1,
    '中部': 2,
    '南部': 3,
    '東部': 4,
    '外島': 5
}

# 3. Create the 'Order' column based on the Region
unit_region_df['Order'] = unit_region_df['Region'].map(order_map)

# 4. Sort by the Order column
unit_region_df = unit_region_df.sort_values(by='Order').reset_index(drop=True)

# 5. Optional: Clean up column names for the final table
unit_region_df.columns = ['Judicial_Unit', 'Region', 'Order']

# Check if any unit appears more than once
duplicates = unit_region_df['Judicial_Unit'].value_counts()
conflict_units = duplicates[duplicates > 1]

if not conflict_units.empty:
    print("Warning: These units are mapped to multiple regions:")
    print(conflict_units)
else:
    print("Success: Every Judicial Unit has a unique Region mapping.")

Success: Every Judicial Unit has a unique Region mapping.


In [16]:
#######         Latitude and Longitude for each city        #########

city_coords = {
    '臺北市': [25.0330, 121.5654], '新北市': [25.0120, 121.4657],
    '桃園市': [24.9936, 121.3010], '臺中市': [24.1477, 120.6736],
    '臺南市': [22.9997, 120.2270], '高雄市': [22.6273, 120.3014],
    '基隆市': [25.1283, 121.7419], '新竹市': [24.8138, 120.9675],
    '嘉義市': [23.4810, 120.4473], '新竹縣': [24.8387, 121.0177],
    '苗栗縣': [24.5601, 120.8209], '彰化縣': [24.0511, 120.5161],
    '南投縣': [23.9159, 120.6810], '雲林縣': [23.7092, 120.4313],
    '嘉義縣': [23.4518, 120.2559], '屏東縣': [22.6730, 120.4862],
    '宜蘭縣': [24.7021, 121.7377], '花蓮縣': [23.9871, 121.6016],
    '臺東縣': [22.7554, 121.1505], '澎湖縣': [23.5711, 119.5793],
    '金門縣': [24.4498, 118.3737], '連江縣': [26.1507, 119.9289]
}

# 1. Count cases per city
map_data = df['rDeathCity'].value_counts().reset_index()
map_data.columns = ['City', 'Case_Count']

# 2. Add Lat/Lon columns
map_data['Lat'] = map_data['City'].map(lambda x: city_coords.get(x, [0,0])[0])
map_data['Lon'] = map_data['City'].map(lambda x: city_coords.get(x, [0,0])[1])

# Remove any that didn't match (safety check)
map_data = map_data[map_data['Lat'] != 0]

In [17]:
date_threshold = pd.to_datetime('2025-07-22').date()

east_pingtung_df = df[
    ((df['Region'] == '東部') | (df['rDeathCity'] == '屏東縣')) & 
    (df['DeathDate'] > date_threshold)
].reset_index(drop=True)

# Peek at the results
print(f"Rows found after July 22, 2025: {len(east_pingtung_df)}")
display(east_pingtung_df)

Rows found after July 22, 2025: 17


,rSerial,rID,rSex,rDead,rSage,rEage,rAgeRange,rDeathCity,rDeathAddr,rPlaceOfDiscovery,...,rUnitID,rUnitName,rNoteUnit,rECaseNo,rHandlingUnit,Reason,DeathDate,WeekdayNum,Weekday,Region
0,450663,20260313010046311,男,不詳(疑似劉鈞),NaN,NaN,不詳,屏東縣,屏東縣牡丹鄉平156線7.15公里處,屏東縣屏東縣牡丹鄉平156線7.15公里處,...,PTN,臺灣屏東地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2026-03-11,2.0,星期三,南部
1,449534,P121597029,男,郭益村,NaN,NaN,不詳,屏東縣,其他屏東縣潮州鎮富春里潮州路300號,其他其他屏東縣潮州鎮富春里潮州路300號,...,PTN,臺灣屏東地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2026-02-21,5.0,星期六,南部
2,449493,20260221010044741,女,不詳(疑似簡妤芹),40.0,50.0,約 40 ～ 50 歲,屏東縣,屏東縣屏東縣滿州鄉里德村(22。02’02.0”N 120。 53’58.9”E),屏東縣屏東縣屏東縣滿州鄉里德村(22。02’02.0”N 120。 53’58.9”E),...,PTN,臺灣屏東地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2026-02-15,6.0,星期日,南部
3,450939,20260318010044896,不詳,不詳,NaN,NaN,不詳,臺東縣,臺東縣臺東市中華路四段513巷底(靠近豐里雷達站)海邊沙灘岸邊,臺東縣臺東縣臺東市中華路四段513巷底(靠近豐里雷達站)海邊沙灘岸邊,...,TTN,臺灣臺東地方檢察署,NaN,NaN,NaN,不明原因死亡。 已白骨化。,2026-02-12,3.0,星期四,東部
4,446987,U121326329,男,疑林世崎,40.0,50.0,約 40 ～ 50 歲,花蓮縣,花蓮縣新城鄉193縣道3.5公里往東100公尺防風林處,花蓮縣花蓮縣新城鄉193縣道3.5公里往東100公尺防風林處,...,HLN,臺灣花蓮地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2026-01-01,3.0,星期四,東部
5,443937,20251106010045562,不詳,不詳,NaN,NaN,不詳,花蓮縣,花蓮縣吉安鄉仁里五街94號,花蓮縣花蓮縣吉安鄉仁里五街94號,...,HLN,臺灣花蓮地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2025-11-05,2.0,星期三,東部
6,443938,20251106010045672,男,不詳(疑似林彥成),NaN,NaN,不詳,屏東縣,屏東縣麟洛鄉鐵道橋下自行車道旁灌溉溝渠,屏東縣屏東縣麟洛鄉鐵道橋下自行車道旁灌溉溝渠,...,PTN,臺灣屏東地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2025-11-05,2.0,星期三,南部
7,442581,20251013010043815,不詳,不詳,NaN,NaN,不詳,花蓮縣,花蓮縣富里鄉六十石山山區(雲閔高支219分7003841FB9600電桿)某空屋前山溝,花蓮縣花蓮縣富里鄉六十石山山區(雲閔高支219分7003841FB9600電桿)某空屋前山溝,...,HLN,臺灣花蓮地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2025-10-12,6.0,星期日,東部
8,441275,20250918010045003,男,不詳,NaN,NaN,不詳,臺東縣,臺東縣臺東縣鹿野鄉綺麗渡假村後方河灘地處,臺東縣臺東縣臺東縣鹿野鄉綺麗渡假村後方河灘地處,...,TTN,臺灣臺東地方檢察署,NaN,NaN,NaN,暫冰存。,2025-09-12,4.0,星期五,東部
9,440850,H100152409,男,劉承怡,70.0,80.0,約 70 ～ 80 歲,花蓮縣,花蓮縣秀林鄉富世村臺8線175.5公里處溪床,花蓮縣花蓮縣秀林鄉富世村臺8線175.5公里處溪床,...,HLN,臺灣花蓮地方檢察署,NaN,NaN,NaN,解剖鑑定中。,2025-09-08,0.0,星期一,東部


In [18]:
from itables import show
import itables.options as opt

# 1. Setup the table to have "Slicers" (column filters) at the top
opt.column_filters = "footer"
opt.maxBytes = 0  # Allow large datasets
opt.classes = ["display", "table", "table-striped", "table-hover"]
opt.lengthMenu = [10, 20, 50]
opt.pageLength = 20

# Add the 'colvis' button to your layout

opt.buttons = [
    {"extend": "pageLength"},
    {"extend": "colvis", "text": "Columns Visibility"},
    {"extend": "csvHtml5", "text": "Export CSV"},
    {"extend": "excelHtml5", "text": "Export Excel"}
]

opt.layout = {
    "topStart": "buttons",
    "topEnd": "search"
}

east_pingtung_df.rename(columns={
  'rID': '編號', 
  'rSex': '性別', 
  'rDead': '姓名', 
  'rAgeRange': '年齡範圍',
  'rDeathCity': '縣市', 
  'rPlaceOfDiscovery': '發現地', 
  'DeathDate': '發現日期',
  'rSize': '身材描述',
  'rHeightRange': '身高', 
  'rBody': '身體特徵', 
  'rDress': '衣著特徵', 
  'rThings': '隨身物品', 
  'rDeathType': '死亡方式',
  'Reason': '死亡原因', 
  'rPlace': '存放地', 
  'rUnitName': '承辦檢察署', 
  'rNoteUnit': '報驗機關', 
  'rECaseNo': 'e化案號', 
  'rHandlingUnit': '承辦單位',
  'Region': '區域',
}, inplace=True)

# Define which columns to hide by their names
cols_to_show = ['發現日期', '編號', '性別', '姓名', '年齡範圍',
       '區域', '發現縣市', '發現地', '死亡原因',
       '身材描述', '身高', '身體特徵', '衣著特徵',
       '隨身物品', '死亡方式', '存放地', '承辦檢察署', '報驗機關',
       '承辦單位', 'e化案號'  
       ]

# 2. Display your cleaned dataframe
show(east_pingtung_df)

In [19]:
df.to_csv('csv/death_full.csv', index=False, encoding='utf-8-sig')

df_chinese_column = df.rename(columns={
  'rSerial': '序號', 
  'rID': '編號', 
  'rSex': '性別', 
  'rDead': '姓名', 
  'rSage': '年齡範圍下限', 
  'rEage': '年齡範圍上限', 
  'rAgeRange': '年齡範圍',
  'rDeathCity': '縣市', 
  'rDeathAddr': '發現地址', 
  'rPlaceOfDiscovery': '發現地', 
  'rDteDeathYYY': '發現年',
  'rDteDeathMM': '發現月', 
  'rDteDeathDD': '發現日', 
  'DeathDate': '發現日期',
  'rTimeOfDiscovery': '發現日期民國', 
  'rSize': '身材描述',
  'rSHeight': '身高下限',
  'rEHeight': '身高上限', 
  'rHeightRange': '身高', 
  'rBody': '身體特徵', 
  'rDress': '衣著特徵', 
  'rThings': '隨身物品', 
  'rDeathType': '死亡方式',
  'Reason': '死亡原因', 
  'rPlace': '存放地', 
  'rUnitID': '承辦檢察署ID',
  'rUnitName': '承辦檢察署', 
  'rNoteUnit': '報驗機關', 
  'rECaseNo': 'e化案號', 
  'rHandlingUnit': '承辦單位',
  'Region': '區域',
  'Weekday': '周日',
  'WeekdayNum': '周日數字'
})

# Saves directly to your project folder
df_chinese_column.to_csv('csv/death_full_chinese_column.csv', index=False, encoding='utf-8-sig')

In [27]:
df['發現日期']

0       2026-03-29
1       2026-03-23
2       2026-03-16
3       2026-03-15
4       2026-03-14
           ...    
2822           NaN
2823           NaN
2824           NaN
2825           NaN
2826           NaN
Name: 發現日期, Length: 2827, dtype: object

In [ ]:
### Notification testing
import os
import requests
import pandas as pd
from dotenv import load_dotenv

df = pd.read_csv('csv/death_full_chinese_column.csv')

# 1. Define your target list
target_counties = ['宜蘭縣', '花蓮縣', '臺東縣', '屏東縣']

df['發現日期'] = pd.to_datetime(df['發現日期']).dt.date

# 2. Create the mask
# We ensure rDeathDate is in date format to match your threshold
mask = (df['發現日期'] > pd.to_datetime('2025-07-22').date()) & \
       (df['縣市'].isin(target_counties))

filtered_df = df[mask]

load_dotenv()

def send_telegram_msg(message):
    token = os.getenv("TOKEN")
    chat_ids = os.getenv("CHAT_ID").split(',')
    for cid in chat_ids:
       url = f"https://api.telegram.org/bot{token}/sendMessage"
       payload = {"chat_id": cid.strip(), "text": message}
       requests.post(url, data=payload)


# 1. Calculate the current count of your specific counties
current_count = len(filtered_df)
count_file = "last_count.txt"

# 2. Read the previous count (if it exists)
if os.path.exists(count_file):
    with open(count_file, "r") as f:
       content = f.read().strip()
        # If the file is empty, default to 0
       last_recorded_count = int(content) if content else 0
else:
    last_recorded_count = 0

# 3. Notification Logic
if current_count > last_recorded_count:
    new_records = current_count - last_recorded_count
    message = f"🚨 Found {new_records} new records in 宜/花/東/屏! Total: {current_count} (Previous: {last_recorded_count})"
    send_telegram_msg(message)
    
    # 4. UPDATE the file so the next run knows the new baseline
    with open(count_file, "w") as f:
        f.write(str(current_count))
    print(f"Count updated to {current_count}")
else:
    print("No new records found.")


Count updated to 25
